# FAHR-FaceAge 얼굴 나이 추정 (사진 폴더 → CSV)

얼굴 붓기 지표 프로젝트 **트랙 B**용 노트북입니다. 사진 폴더를 올리면 각 사진의 얼굴 나이를 추정해 `파일명, 추정 나이`를 CSV로 저장합니다.

**처리 순서** (FAHR-Face 저장소 README와 Hugging Face 모델 카드 기준)
1. RetinaFace(`retina-face==0.0.17`, 저장소 `other/face_crop_align.py`와 동일)로 얼굴 검출 → **얼굴이 정확히 1개인 사진만** 사용
2. `RetinaFace.extract_faces(align=True)`로 눈 위치 기준 회전 정렬 + 크롭
3. 112×112 RGB로 리사이즈, ImageNet 평균·표준편차로 정규화
4. `fahr_faceage.pth`(Hugging Face `fridolin22/FAHR-Face`)로 나이 회귀

**사용 방법**: 메뉴 `런타임 → 모두 실행`. 폰 앱에서 "사진 공유 → Google Drive"로 `faceage_photos` 폴더에 보냈다면 4번 셀에서 `INPUT_MODE = "drive"`를 고르세요(PC로 내려받을 필요 없음). GPU가 없어도 됩니다.

**걸리는 시간(정상 범위)**: 새 세션마다 ① 패키지 설치 1~3분 ② 가중치 다운로드 수 분 ③ 첫 사진에서 얼굴 검출 모델(약 119 MB) 다운로드. 그 뒤 사진 1장당 CPU에서 약 2~5초. 5번 셀에서 `[1/10] 파일명: 나이`처럼 한 줄씩 출력되면 정상 진행 중입니다.
**더 빠르게**: 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU`를 고르면 검출·추론이 빨라집니다(무료 계정은 사용 시간 제한이 있음).

**파이프라인 점검(편향 원인 구분용)**: 7번 셀의 정렬된 얼굴이 눈·코·입이 가운데 오게 반듯이 잘렸는지 확인하세요. 본인의 **촬영 연도를 아는 예전 사진**(예: 5년·10년 전)을 `past_YYYY/` 폴더로 넣으면, 나이가 바뀔 때 추정값도 그만큼 바뀌는지(모델이 본인 얼굴의 노화를 따라가는지) 볼 수 있습니다.

**사진 준비 요령**
- JPG/PNG만 지원합니다(아이폰 HEIC는 JPG로 변환).
- 조건별로 **하위 폴더**를 만들어 zip으로 올리면 폴더 이름이 `condition` 열이 되어 조건별 평균·SD를 자동으로 계산합니다. 예: `standard/`, `casual_selfie/`, `morning/`, `evening/`, `lamp_warm/`
- 붓기 앱의 "표준 사진 저장"으로 받은 파일(`ff_..._s1_...jpg` ~ `_s10_`)은 파일명에서 시각·턱 높이·좌우·거리·안경 여부·해상도를 읽고, **같은 측정의 사진(현재 앱은 10장, 이전 버전은 3장)을 자동으로 묶어 평균**합니다. 파일명이 바뀌지 않게 메신저 대신 Google Drive나 USB로 옮기세요.
- 앱 사진이 아닌 경우(일반 셀카 등) 측정 단위로 묶으려면 `조건/측정이름/사진.jpg`처럼 폴더를 한 단계 더 만드세요. 예: `casual_selfie/1회차/`, `casual_selfie/2회차/`. 이렇게 하지 않으면 사진 한 장이 한 측정으로 계산됩니다.

**주의**
- 모델 가중치 라이선스: **CC BY-NC-SA 4.0 (비상업·학술 연구용)**. 코드: Apache 2.0. 상용 앱에는 그대로 쓸 수 없습니다.
- 공개 벤치마크(APPA-REAL)에서 평균 절대오차 약 5.1세로 보고된 모델입니다. 여기서는 절대값보다 **같은 사람 반복 측정의 흔들림(SD)**을 보는 것이 목적입니다.
- Colab에 올린 사진은 Google 서버의 임시 런타임에 저장되며 세션이 끝나면 지워집니다. **IRB 승인 전에는 본인 사진만** 사용하세요.

**1번 셀이 5분 넘게 돌기만 하면**: 화면 오른쪽 위가 `연결 중`이면 아직 런타임이 배정되지 않은 것입니다(무료 계정은 혼잡 시간대에 늦어짐). `런타임 → 런타임 연결 해제 및 삭제` 후 다시 `모두 실행`하세요. 휴대폰보다 PC 브라우저가 안정적입니다.

## 1. 환경 준비

In [ ]:
# FAHR-Face 저장소(모델 클래스) + 전처리 패키지 설치
# 각 단계마다 [1/5]처럼 진행 상황을 출력합니다. 보통 1~3분이면 끝납니다.
import os, sys, time, importlib.util, importlib.metadata as md
T0 = time.time()
def step(msg): print(f"[{time.time()-T0:5.0f}초] {msg}", flush=True)

# retina-face 0.0.17은 Keras 2 기준입니다. TensorFlow를 import하기 전에 Keras 2(tf-keras)를 쓰도록 지정합니다.
if "tensorflow" in sys.modules and os.environ.get("TF_USE_LEGACY_KERAS") != "1":
    raise SystemExit("⚠ 이 런타임에서 TensorFlow가 이미 Keras 3로 로드되었습니다. 메뉴 '런타임 → 세션 다시 시작'을 누른 뒤 '모두 실행'을 다시 하세요.")
os.environ["TF_USE_LEGACY_KERAS"] = "1"

def have(pkg):
    try: return md.version(pkg)
    except md.PackageNotFoundError: return None

step("1/5 FAHR-Face 저장소 받기")
# 결과 재현을 위해 시험한 커밋으로 고정합니다(저장소가 나중에 바뀌어도 같은 코드 사용)
FAHR_COMMIT = "a49cb07f3e15858eab5024b00e1796a8b3cfef9e"
if not os.path.exists("/content/FAHR-Face/README.md"):
    !rm -rf /content/FAHR-Face && git clone -q https://github.com/AIM-Harvard/FAHR-Face.git /content/FAHR-Face
!git -C /content/FAHR-Face checkout -q {FAHR_COMMIT}

# 의존성 계산(pip resolver) 때문에 오래 멈추는 일을 막으려고, 이미 Colab에 있는 패키지는 건드리지 않고
# 빠진 것만 --no-deps로 설치합니다.
step("2/5 retina-face, huggingface_hub 설치")
if not have("retina-face"):
    !pip install -q --no-deps "retina-face==0.0.17"
need = [p for p in ["gdown", "huggingface_hub", "opencv-python-headless"] if not have(p)]
if "opencv-python-headless" in need and (have("opencv-python") or importlib.util.find_spec("cv2")):
    need.remove("opencv-python-headless")
need_s = " ".join(need)
if need:
    !pip install -q {need_s}

step("3/5 TensorFlow 확인")
tf_ver = have("tensorflow")
if not tf_ver:
    step("   TensorFlow가 없어 설치합니다(약 600 MB, 2~4분). 진행 막대가 보이면 정상입니다.")
    !pip install --progress-bar on "tensorflow<2.21"
    tf_ver = have("tensorflow")
tf_mm = ".".join(tf_ver.split(".")[:2])

step(f"4/5 tf-keras {tf_mm} 확인 (TensorFlow {tf_ver})")
kv = have("tf-keras")
if not kv or ".".join(kv.split(".")[:2]) != tf_mm:
    !pip install -q --no-deps "tf-keras~={tf_mm}.0"

step("5/5 라이브러리 불러오기 (TensorFlow import에 30초 정도 걸릴 수 있음)")
sys.path.insert(0, "/content/FAHR-Face")
import torch, transformers
import tensorflow as tf
print("torch", torch.__version__, "| transformers", transformers.__version__, "| tensorflow", tf.__version__)
print("Keras 엔진:", tf.keras.Model.__module__.split(".")[0], "(tf_keras 이어야 정상)")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
step("✅ 환경 준비 완료")

## 2. 모델 가중치 받기 (`fahr_faceage.pth`, 로그인 불필요)

In [ ]:
from huggingface_hub import hf_hub_download
# 가중치 개정(revision)을 고정하려면 HF_REVISION에 커밋 해시를 적으세요. 비워 두면 최신 → 실제 받은 개정과 SHA-256은 결과의 provenance 파일에 기록됩니다
HF_REVISION = ""   # @param {type:"string"}
CKPT = hf_hub_download("fridolin22/FAHR-Face", "fahr_faceage.pth", revision=HF_REVISION or None)
print(CKPT, f"{os.path.getsize(CKPT)/1e6:.0f} MB")

## 3. 분석 함수

In [ ]:
import os, re, glob, zipfile, hashlib, warnings
import numpy as np, pandas as pd, torch
from PIL import Image
from torchvision import transforms
from transformers import ViTConfig
from ViTForAgeEstimation import ViTForAgeRegression   # FAHR-Face 저장소의 원본 모델 클래스
from retinaface import RetinaFace

IMG_EXT = (".jpg", ".jpeg", ".png")
TRANSFORM = transforms.Compose([
    transforms.Resize((112, 112)),          # README/model card: 112x112 RGB
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
APP_RE = re.compile(r"^ff_(?P<ts>\d{8}T\d{6})_(?P<session>[a-z]+)(?:_s(?P<shot>\d+))?_pitch(?P<pitch>-?[\d.]+)_yaw(?P<yaw>-?[\d.]+)_roll(?P<roll>-?[\d.]+)_dist(?P<dist>[\d.]+)_gl(?P<glasses>[01])(?:_src(?P<src>(?:hi|vid)\d+x\d+))?")

# 체크포인트는 transformers 4.x의 ViT 층 이름으로 저장되어 있음. 최신 transformers(5.x)는 층 이름이 바뀌었으므로
# 현재 설치된 버전의 이름에 맞춰 변환한다(가중치 값은 그대로).
_OLD2NEW = [
    (r"^vit\.encoder\.layer\.(\d+)\.attention\.attention\.query\.", r"vit.layers.\1.attention.q_proj."),
    (r"^vit\.encoder\.layer\.(\d+)\.attention\.attention\.key\.",   r"vit.layers.\1.attention.k_proj."),
    (r"^vit\.encoder\.layer\.(\d+)\.attention\.attention\.value\.", r"vit.layers.\1.attention.v_proj."),
    (r"^vit\.encoder\.layer\.(\d+)\.attention\.output\.dense\.",   r"vit.layers.\1.attention.o_proj."),
    (r"^vit\.encoder\.layer\.(\d+)\.intermediate\.dense\.",         r"vit.layers.\1.mlp.fc1."),
    (r"^vit\.encoder\.layer\.(\d+)\.output\.dense\.",               r"vit.layers.\1.mlp.fc2."),
    (r"^vit\.encoder\.layer\.(\d+)\.layernorm_",                      r"vit.layers.\1.layernorm_"),
]
def _convert_keys(sd, model_keys):
    if all(k in model_keys for k in sd):           # 이미 이름이 같음(구버전 transformers)
        return sd
    out = {}
    for k, v in sd.items():
        nk = k
        for pat, rep in _OLD2NEW:
            nk2 = re.sub(pat, rep, nk)
            if nk2 != nk: nk = nk2; break
        out[nk] = v
    return out

def load_age_model(ckpt_path, device):
    config = ViTConfig(image_size=112, num_hidden_layers=12, num_attention_heads=12)
    model = ViTForAgeRegression(config)
    # 안전한 방식(weights_only=True, 텐서·기본 자료형만 허용)으로 먼저 연다.
    # 체크포인트에 그 밖의 객체가 있어 실패하면, 출처(Hugging Face fridolin22/FAHR-Face)와 SHA-256을 기록한 뒤에만 일반 방식으로 연다.
    global LOAD_MODE
    try:
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=True); LOAD_MODE = "weights_only=True"
    except Exception as e:
        warnings.warn(f"weights_only=True로 열 수 없어 일반 방식으로 엽니다(출처·해시는 provenance에 기록): {str(e)[:120]}")
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False); LOAD_MODE = "weights_only=False(fallback)"
    sd = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
    sd = _convert_keys(sd, set(model.state_dict().keys()))
    missing, unexpected = model.load_state_dict(sd, strict=False)
    bad_missing = [k for k in missing if not k.endswith("position_ids")]
    bad_unexp = [k for k in unexpected if not k.endswith("position_ids")]
    if bad_missing or bad_unexp:
        raise RuntimeError(f"체크포인트와 모델 구조가 맞지 않습니다. missing={bad_missing[:5]} unexpected={bad_unexp[:5]}")
    return model.to(device).eval()

LOAD_MODE = None
FAHR_COMMIT = "a49cb07f3e15858eab5024b00e1796a8b3cfef9e"   # 시험한 FAHR-Face 저장소 커밋(2026-11-11)

def sha256(path, n=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(n), b""): h.update(b)
    return h.hexdigest()

def provenance(ckpt_path, repo_dir="/content/FAHR-Face"):
    """같은 사진의 결과가 나중에 달라졌을 때 원인을 찾을 수 있도록 버전 정보를 남긴다."""
    import subprocess, transformers, torchvision
    try: commit = subprocess.run(["git", "-C", repo_dir, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    except Exception: commit = ""
    snap = os.path.basename(os.path.dirname(os.path.realpath(ckpt_path))) if "snapshots" in os.path.realpath(ckpt_path) else ""
    try:
        import importlib.metadata as md; rf = md.version("retina-face")
    except Exception: rf = ""
    return {"fahr_repo_commit": commit, "hf_snapshot": snap, "ckpt_sha256": sha256(ckpt_path), "ckpt_load": LOAD_MODE,
            "torch": torch.__version__, "torchvision": torchvision.__version__, "transformers": transformers.__version__, "retina_face": rf}

def list_images(root):
    out = []
    for p in sorted(glob.glob(os.path.join(root, "**", "*"), recursive=True)):
        if p.lower().endswith(IMG_EXT) and "/__MACOSX/" not in p and not os.path.basename(p).startswith("._"):
            out.append(p)
    return out

MAX_SIDE = 1600   # 큰 사진(10MP 등)은 검출 전에 긴 변 1600px로 축소: 속도 약 4~5배. 모델 입력은 어차피 112x112

def _load_bgr(path):
    import cv2
    img = cv2.imread(path)                  # EXIF 회전 반영, BGR
    if img is None: raise ValueError("이미지를 읽을 수 없음")
    h, w = img.shape[:2]; s = MAX_SIDE / max(h, w)
    if s < 1: img = cv2.resize(img, (int(w*s), int(h*s)), interpolation=cv2.INTER_AREA)
    return img, (1/s if s < 1 else 1.0)

def _align_crop(img, identity):
    """retina-face 0.0.17 extract_faces(align=True)와 같은 계산 (검출을 한 번만 하려고 그대로 옮김)."""
    from retinaface.commons import postprocess
    x1, y1, x2, y2 = identity["facial_area"]; x, y, w, h = x1, y1, x2 - x1, y2 - y1
    lm = identity["landmarks"]
    aligned, ang, direc = postprocess.alignment_procedure(img=img, left_eye=lm["right_eye"], right_eye=lm["left_eye"], nose=lm["nose"])
    rx1, ry1, rx2, ry2 = postprocess.rotate_facial_area((x, y, x + w, y + h), ang, direc, (img.shape[0], img.shape[1]))
    return aligned[int(ry1):int(ry2), int(rx1):int(rx2)][:, :, ::-1]   # BGR -> RGB

def detect_crop(path, threshold=0.9):
    """README 방식: 얼굴이 정확히 1개일 때만 RetinaFace로 정렬(align=True) 후 크롭."""
    img, scale = _load_bgr(path)
    resp = RetinaFace.detect_faces(img, threshold=threshold)
    n = len(resp) if isinstance(resp, dict) else 0
    if n != 1:
        return None, n, None
    ident = list(resp.values())[0]
    crop = _align_crop(img, ident)
    info = dict(ident); info["facial_area"] = [int(round(v*scale)) for v in ident["facial_area"]]   # 원본 픽셀 기준
    if crop.size == 0:
        return None, n, info
    return Image.fromarray(np.ascontiguousarray(crop)), n, info

@torch.no_grad()
def predict(model, pil_img, device):
    """원 저장소 run.py와 같은 방식: 정렬된 크롭을 112x112로 리사이즈(종횡비 유지 안 함) → ImageNet 정규화.
    반환: (원본 추정, 좌우 반전과의 평균). 좌우 반전 평균은 시험 시 증강(TTA) 결과로 별도 열에 둔다."""
    img = pil_img.convert("RGB")
    x = torch.stack([TRANSFORM(img), TRANSFORM(img.transpose(Image.FLIP_LEFT_RIGHT))]).to(device)
    a = model(x).age_output.reshape(-1).float().cpu().numpy()
    return float(a[0]), float(a.mean())

def parse_meta(path, root):
    """condition = 첫 번째 하위 폴더. measurement_id = 앱 사진이면 촬영 시각(한 측정의 사진들이 같은 값),
    아니면 두 번째 하위 폴더(직접 묶은 경우), 그것도 없으면 사진 한 장이 한 측정."""
    name = os.path.basename(path)
    parts = [x for x in os.path.relpath(os.path.dirname(path), root).split(os.sep) if x not in (".", "")]
    m = APP_RE.match(name)
    # 폴더 없이 사진만 올린 경우(폰에서 바로 업로드): 앱 표준 사진과 그 밖의 사진을 자동으로 나눔
    meta = {"file": name, "condition": parts[0] if parts else ("app_standard" if m else "other_photos")}
    if m:
        d = m.groupdict()
        if not parts:   # 하위 폴더 없이 Drive 폴더에 쌓인 앱 사진: 날짜·시간대별로 조건을 나눠 세션이 섞이지 않게 함
            meta["condition"] = f'app_{d["ts"][:8]}_{d["session"]}'
        meta["app_date"] = d["ts"][:8]
        meta.update({"measurement_id": f'{d["ts"]}_{d["session"]}', "shot": int(d["shot"]) if d.get("shot") else 1,
                     "app_timestamp": d["ts"], "app_session": d["session"], "app_pitch": float(d["pitch"]),
                     "app_yaw": float(d["yaw"]), "app_roll": float(d["roll"]), "app_dist": float(d["dist"]),
                     "app_glasses": int(d["glasses"]), "app_src": d.get("src") or ""})
    elif len(parts) >= 2:
        meta.update({"measurement_id": "/".join(parts[1:]), "shot": None})
    else:
        meta.update({"measurement_id": os.path.splitext(name)[0], "shot": None})
    return meta

def run_folder(root, model, device, crop_dir=None, threshold=0.9, log=print):
    rows = []
    paths = list_images(root)
    seen_hash, seen_shot = {}, {}
    for i, p in enumerate(paths, 1):
        meta = parse_meta(p, root)
        # 같은 측정을 Drive에 두 번 공유하면 "파일 (1).jpg"가 생김 → 같은 내용 또는 같은 (측정, 장 번호)는 한 번만 사용
        h = sha256(p); key = (meta["condition"], meta["measurement_id"], meta.get("shot")) if meta.get("shot") else None
        dup_of = seen_hash.get(h) or (seen_shot.get(key) if key else None)
        if dup_of:
            meta.update({"status": "duplicate", "reason": f"중복: {dup_of}"}); rows.append(meta)
            log(f"[{i}/{len(paths)}] {meta['file']}: 중복이라 건너뜀"); continue
        seen_hash[h] = meta["file"]
        if key: seen_shot[key] = meta["file"]
        try:
            face, n, info = detect_crop(p, threshold)
            if face is None:
                meta.update({"status": "skipped", "reason": "얼굴 없음" if n == 0 else (f"얼굴 {n}개" if n > 1 else "크롭 실패"), "n_faces": n})
            else:
                a0, a_flip = predict(model, face, device)
                meta.update({"status": "ok", "n_faces": 1, "predicted_age": round(a0, 2), "predicted_age_flipavg": round(a_flip, 2),
                             "det_score": round(float(info.get("score", np.nan)), 4),
                             "face_w_px": int(info["facial_area"][2] - info["facial_area"][0]),
                             "face_h_px": int(info["facial_area"][3] - info["facial_area"][1])})
                if crop_dir:
                    sub = os.path.join(crop_dir, meta["condition"]) if meta["condition"] else crop_dir
                    os.makedirs(sub, exist_ok=True)
                    face.save(os.path.join(sub, os.path.splitext(meta["file"])[0] + ".png"))
        except Exception as e:
            meta.update({"status": "error", "reason": str(e)[:200]})
        rows.append(meta)
        log(f"[{i}/{len(paths)}] {meta['file']}: {meta.get('predicted_age', meta.get('reason'))}")
    cols = ["file", "condition", "measurement_id", "shot", "predicted_age", "predicted_age_flipavg", "status", "reason", "n_faces", "det_score", "face_w_px", "face_h_px",
            "app_date", "app_timestamp", "app_session", "app_pitch", "app_yaw", "app_roll", "app_dist", "app_glasses", "app_src"]
    df = pd.DataFrame(rows)
    return df[[c for c in cols if c in df.columns]]

def summarize(df):
    """사진 단위 요약: 조건별 사진 한 장의 평균·SD."""
    ok = df[df["status"] == "ok"].copy()
    if ok.empty: return pd.DataFrame()
    ok["condition"] = ok["condition"].replace("", "(전체)")
    s = ok.groupby("condition")["predicted_age"].agg(n_photos="count", mean="mean", sd="std", min="min", max="max").round(2)
    s["range"] = (s["max"] - s["min"]).round(2)
    return s.reset_index()

MIN_VALID_FRAC = 0.8   # 측정당 최소 유효 사진 비율(10장 중 8장). 미달 측정은 valid=False → 재촬영 권장

def _trimmed(v):
    v = np.sort(np.asarray(v, float))
    return float(v[1:-1].mean()) if len(v) >= 5 else float(v.mean())   # 5장 이상이면 최솟값·최댓값 1개씩 제외

def per_measurement(df, col="predicted_age"):
    """측정 단위 표: 같은 측정(앱 1회 = 10장)의 평균·중앙값·절사 평균, 측정 내 SD, 유효 장수 판정."""
    ok = df[df["status"] == "ok"].copy()
    if ok.empty: return pd.DataFrame()
    ok["condition"] = ok["condition"].replace("", "(전체)")
    rows = []
    for (c, mid), g in ok.groupby(["condition", "measurement_id"]):
        v = g[col].values
        shots_all = df[(df["condition"] == c) & (df["measurement_id"] == mid)]
        sh = pd.to_numeric(shots_all.get("shot"), errors="coerce") if "shot" in shots_all else pd.Series(dtype=float)
        expected = (10 if sh.max() > 3 else 3) if sh.notna().any() else len(shots_all[shots_all["status"] != "duplicate"])
        rows.append({"condition": c, "measurement_id": mid, "n_photos": len(v), "n_expected": int(expected),
                     "valid": bool(len(v) >= np.ceil(MIN_VALID_FRAC * expected)),
                     "mean_age": v.mean(), "median_age": float(np.median(v)), "trimmed_mean_age": _trimmed(v),
                     "sd_within": v.std(ddof=1) if len(v) > 1 else np.nan, "min": v.min(), "max": v.max()})
    return pd.DataFrame(rows).round(2)

def summarize_measurements(meas):
    """조건별 측정 단위 요약: 측정 평균들 사이의 SD(= 측정 평균을 쓸 때의 반복 오차)."""
    if meas.empty: return pd.DataFrame()
    g = meas[meas["valid"]].groupby("condition") if "valid" in meas and meas["valid"].any() else meas.groupby("condition")
    s = g.agg(n_measurements=("mean_age", "count"), photos_per_measurement=("n_photos", "mean"),
              mean=("mean_age", "mean"), sd_between_measurements=("mean_age", "std"),
              sd_between_median=("median_age", "std"), sd_between_trimmed=("trimmed_mean_age", "std"),
              mean_sd_within=("sd_within", "mean")).round(2)
    return s.reset_index()

# 사진이 너무 작으면(얼굴 폭 < 112px) 업스케일된 얼굴로 추정하게 되어 신뢰도가 떨어집니다.
MIN_FACE_PX = 112
MODEL = load_age_model(CKPT, DEVICE)
PROV = provenance(CKPT)
print("FAHR-FaceAge 로드 완료 |", PROV["ckpt_load"], "| 저장소 커밋", PROV["fahr_repo_commit"][:7], "| 가중치 SHA-256", PROV["ckpt_sha256"][:12])
if PROV["fahr_repo_commit"] and PROV["fahr_repo_commit"] != FAHR_COMMIT:
    print("⚠ 시험한 저장소 커밋과 다릅니다:", FAHR_COMMIT[:7])

## 4. 사진 올리기

- `INPUT_MODE = "upload"`: 실행하면 파일 선택 창이 뜹니다. **zip 1개**(하위 폴더 가능) 또는 **사진 여러 장**을 고르세요.
  - **폴드7에서 이 노트북을 바로 열었다면** 파일 선택 창에서 `다운로드` 폴더의 사진을 여러 장 고르면 됩니다. 옮길 필요가 없습니다.
  - 폴더 없이 사진만 올리면 앱 표준 사진(`ff_...`)은 `app_standard`, 나머지는 `other_photos` 조건으로 자동 분류됩니다.
- `INPUT_MODE = "drive"`: Google Drive 폴더 경로를 `DRIVE_FOLDER`에 적으세요. (예: `/content/drive/MyDrive/faceage_photos`)

In [ ]:
INPUT_MODE = "upload"   # @param ["upload", "drive"]
DRIVE_FOLDER = "/content/drive/MyDrive/faceage_photos"   # @param {type:"string"}

import shutil, zipfile
PHOTO_DIR = "/content/photos"
if INPUT_MODE == "upload":
    from google.colab import files
    shutil.rmtree(PHOTO_DIR, ignore_errors=True); os.makedirs(PHOTO_DIR)
    uploaded = files.upload()
    for name, data in uploaded.items():
        dst = os.path.join(PHOTO_DIR, name)
        with open(dst, "wb") as f: f.write(data)
        if name.lower().endswith(".zip"):
            with zipfile.ZipFile(dst) as z: z.extractall(PHOTO_DIR)
            os.remove(dst)
    # zip이 최상위 폴더 하나로 감싸져 있으면 그 안을 기준으로
    entries = [e for e in os.listdir(PHOTO_DIR) if e != "__MACOSX"]
    if len(entries) == 1 and os.path.isdir(os.path.join(PHOTO_DIR, entries[0])):
        sub = os.path.join(PHOTO_DIR, entries[0])
        if any(os.path.isdir(os.path.join(sub, e)) for e in os.listdir(sub)): PHOTO_DIR = sub
else:
    from google.colab import drive
    drive.mount("/content/drive")
    PHOTO_DIR = DRIVE_FOLDER

imgs = list_images(PHOTO_DIR)
print(f"사진 {len(imgs)}장 발견:", PHOTO_DIR)
for p in imgs[:10]: print(" ", os.path.relpath(p, PHOTO_DIR))

## 5. 얼굴 나이 추정 실행

In [ ]:
CROP_DIR = "/content/aligned_faces"
shutil.rmtree(CROP_DIR, ignore_errors=True)
DETECTION_THRESHOLD = 0.9   # @param {type:"number"}

df = run_folder(PHOTO_DIR, MODEL, DEVICE, crop_dir=CROP_DIR, threshold=DETECTION_THRESHOLD)
if len(df) and (df["status"] == "error").all():
    raise RuntimeError("모든 사진에서 오류가 났습니다. 첫 오류: " + str(df["reason"].iloc[0]) + "\n→ 이 화면을 캡처해 보내 주세요.")
if "face_w_px" in df.columns:
    df["small_face_warning"] = df["face_w_px"].lt(MIN_FACE_PX) & df["status"].eq("ok")

OUT_CSV = "/content/faceage_results.csv"
df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
import json, datetime
PROV["run_time"] = datetime.datetime.now().isoformat(timespec="seconds"); PROV["detection_threshold"] = DETECTION_THRESHOLD
with open("/content/faceage_provenance.json", "w") as f: json.dump(PROV, f, ensure_ascii=False, indent=1)
print("저장:", OUT_CSV)
print(df["status"].value_counts().to_string())
display(df)

## 6. 요약: 사진 단위와 측정 단위

In [ ]:
summary = summarize(df)                         # 사진 한 장 기준
meas = per_measurement(df)                      # 측정(앱 1회 = 10장) 기준
meas_summary = summarize_measurements(meas)
summary.to_csv("/content/faceage_summary.csv", index=False, encoding="utf-8-sig")
meas.to_csv("/content/faceage_measurements.csv", index=False, encoding="utf-8-sig")
meas_summary.to_csv("/content/faceage_measurement_summary.csv", index=False, encoding="utf-8-sig")
print("① 사진 한 장 기준 (조건별)"); display(summary)
print("② 측정별: 평균·중앙값·절사 평균(최솟값·최댓값 제외), valid = 10장 중 8장 이상 분석됨"); display(meas)
if len(meas) and (~meas["valid"]).any():
    print("⚠ 유효 사진 수가 부족한 측정(요약에서 제외, 재촬영 권장):", meas.loc[~meas["valid"], "measurement_id"].tolist())
print("③ 측정 단위 요약(같은 조건 = 같은 날·시간대): sd_between_* = 측정 평균/중앙값/절사 평균들 사이의 SD(반복 오차), mean_sd_within = 한 측정 안 사진들의 SD")
display(meas_summary)
if df.get("small_face_warning") is not None and df["small_face_warning"].any():
    print("⚠ 얼굴 폭이 112px보다 작은 사진:", df.loc[df["small_face_warning"], "file"].tolist())

## 7. 정렬된 얼굴 확인 (전처리가 제대로 됐는지 눈으로 점검)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
ok = df[df["status"] == "ok"].head(24)
if len(ok):
    cols = 6; rows = (len(ok) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2.4))
    for ax in axes.ravel(): ax.axis("off")
    for ax, (_, r) in zip(axes.ravel(), ok.iterrows()):
        sub = os.path.join(CROP_DIR, r["condition"]) if r["condition"] else CROP_DIR
        ax.imshow(Image.open(os.path.join(sub, os.path.splitext(r["file"])[0] + ".png")).resize((112, 112)))
        ax.set_title(f'{r["predicted_age"]:.1f}세\n{(r["condition"] or "")[:12]}', fontsize=8)
    plt.tight_layout(); plt.show()

## 8. 결과 내려받기

In [ ]:
from google.colab import files
files.download(OUT_CSV)
files.download("/content/faceage_summary.csv")
files.download("/content/faceage_measurements.csv")
files.download("/content/faceage_measurement_summary.csv")
files.download("/content/faceage_provenance.json")
# 정렬된 얼굴 이미지도 받으려면 아래 두 줄의 주석을 푸세요 (얼굴 사진이므로 보관에 주의)
# shutil.make_archive("/content/aligned_faces", "zip", CROP_DIR)
# files.download("/content/aligned_faces.zip")

## 출력 열 설명

| 열 | 의미 |
|---|---|
| `file` | 파일명 |
| `condition` | 첫 번째 하위 폴더 이름(조건). 하위 폴더 없이 쌓인 앱 사진은 `app_{날짜}_{am/pm/etc}`로 자동 구분 |
| `measurement_id`, `shot` | 측정 묶음(앱 사진은 촬영 시각, 아니면 두 번째 하위 폴더), 측정 안의 몇 번째 사진 |
| `predicted_age` | FAHR-FaceAge 추정 나이(세). 원 저장소 `run.py`와 같은 전처리(정렬 크롭 → 112×112 리사이즈, 종횡비 유지 안 함) |
| `predicted_age_flipavg` | 원본과 좌우 반전 크롭의 평균(시험 시 증강). 사진 한 장 잡음을 줄이는지 비교용 |
| `status` / `reason` | `ok`, `skipped`(얼굴 0개 또는 2개 이상), `duplicate`(같은 내용 파일 또는 같은 측정·장 번호, 예: Drive에 두 번 공유한 "(1)" 파일), `error` |
| `det_score`, `face_w_px`, `face_h_px` | RetinaFace 검출 점수, 원본에서의 얼굴 크기 |
| `small_face_warning` | 얼굴 폭 112px 미만 |
| `app_*` | 붓기 앱 표준 사진 파일명에서 읽은 촬영 시각·시점(am/pm/etc)·턱 높이·좌우·기울기·거리(얼굴 폭 %)·안경·사진 출처(`app_src`: `hi` = 고해상도 정지 사진, `vid` = 영상 프레임, 뒤는 해상도) |

**참고**: 저장소의 전처리 환경은 Python 3.8 + `retina-face 0.0.17`이고, 모델 카드의 추론 예시는 Python 3.11 + `torch 2.1.2` + `transformers 4.38.2`에서 시험되었습니다. 이 노트북은 Colab 기본 torch·transformers를 씁니다. 체크포인트는 transformers 4.x의 층 이름으로 저장되어 있어, 5.x에서는 3단계에서 층 이름을 자동 변환합니다(4.38.2로 만든 모델과 5.22.2에서 변환해 불러온 모델의 출력 차이 < 1e-6로 확인). 그래도 구조 오류가 나면 1단계 셀에 `!pip -q install "transformers==4.38.2"`를 추가하고 런타임을 다시 시작하세요.

**결과 파일 4개**: `faceage_results.csv`(사진별), `faceage_summary.csv`(조건별 사진 한 장 기준), `faceage_measurements.csv`(측정별 평균), `faceage_measurement_summary.csv`(조건별 측정 단위 반복 오차)

**측정 표의 열**: `n_expected`(앱 10장 또는 이전 버전 3장), `valid`(유효 사진 ≥ 80%), `mean_age`, `median_age`, `trimmed_mean_age`(5장 이상일 때 최솟값·최댓값 1개씩 제외), `sd_within`.

**provenance 파일**: FAHR-Face 저장소 커밋, 가중치 SHA-256과 Hugging Face 개정, 체크포인트 로드 방식(`weights_only=True` 우선), torch·transformers·retina-face 버전.